In [1]:
import earthkit.data as ekd
import earthkit.regrid as ekr
import xarray as xr
import gc
from pathlib import Path
import numpy as np
import glob

In [2]:
# ==========================================================
# User settings
# ==========================================================

init_dates = ["20251228", "20251229", "20251230","20260107","20260108","20260109","20260110","20260121","20260122","20260122","20260123","20260124","20260204","20260205","20260206","20260207"]   # <-- list of YYYYMMDD
init_hour  = "00"

variables = ["tos"]

outdir = "/g/data/dx2/cl7720/AIFS/AIFS_marine/netcdf"
Path(outdir).mkdir(parents=True, exist_ok=True)

In [3]:
def convert(init_date):
    """Convert all variables for one initialisation date."""

    # --- find the input file (job-id part of the name varies) ---
    pattern = (
        f"/g/data/dx2/hc7988/ECMWF/ocean/"
        f"rd:j6aw:oper:{init_date}:{init_hour}00:g/fc:o2d.*.data"
    )
    matches = glob.glob(pattern)
    if not matches:
        print(f"  !! no file found for {init_date}, skipping")
        return
    infile = matches[0]

    # --- parse date for the time axis ---
    init_year  = int(init_date[0:4])
    init_month = int(init_date[4:6])
    init_day   = int(init_date[6:8])
    start = np.datetime64(f"{init_year}-{init_month:02}-{init_day:02}T{init_hour}")

    # --- per-date output directory ---
    date_outdir = Path(outdir) / init_date # e.g. .../data/20260128
    date_outdir.mkdir(parents=True, exist_ok=True)

    for var in variables:

        print(f"\n[{init_date}] Processing {var}")

        ds = ekd.from_source("file", infile).sel(shortName=var)

        slices = []
        for i in range(len(ds)):
            f_out = ekr.regrid(
                ds[i],
                grid={"grid": [0.25, 0.25]},
                interpolation="linear",
            )
            slices.append(f_out.to_xarray())
            del f_out
            gc.collect()

        da = xr.concat(slices, dim="step")
        del slices
        gc.collect()

        # remove earthkit's non-serializable metadata
        da.attrs.pop("_earthkit", None)
        for v in da.variables:
            da[v].attrs.pop("_earthkit", None)

        # reconstruct valid_time (tp starts at +6h, others at +0h)
        offset = 1 if var == "tp" else 0
        valid_time = start + (np.arange(da.sizes["step"]) + offset) * np.timedelta64(6, "h")
        da = da.assign_coords(valid_time=("step", valid_time))

        xvar = list(da.data_vars)[0]
        
        outfile = f"{date_outdir}/{init_date}T{init_hour}_{var}.nc"

        da.to_netcdf(
            outfile,
            encoding={xvar: {"zlib": True, "complevel": 4}},
        )
        print("  saved:", outfile)

        del da, ds
        gc.collect()

In [4]:
# ==========================================================
# Loop over all init dates
# ==========================================================

for d in init_dates:
    print(f"\n======== {d} ========")
    convert(d)

print("\nAll dates finished.")


======== 20251228 ========

[20251228] Processing tos
CACHE-MANAGER mir/weights, /tmp/cache does not exist
CACHE-MANAGER mir/weights, /tmp/cache created
Cache file /tmp/cache/mir/weights/18/linear/R320-52955330f86b75ed26f8461efb283c90-89.7849:0:-89.7849:359.719/LL-0.25x0.25-90:0:-90:359.75-c0f58519efefe08dc05934e99a59dce4.mat does not exist
Creating cache file /tmp/cache/mir/weights/18/linear/R320-52955330f86b75ed26f8461efb283c90-89.7849:0:-89.7849:359.719/LL-0.25x0.25-90:0:-90:359.75-c0f58519efefe08dc05934e99a59dce4.mat
CacheManager creating file /tmp/cache/mir/weights/18/linear/R320-52955330f86b75ed26f8461efb283c90-89.7849:0:-89.7849:359.719/LL-0.25x0.25-90:0:-90:359.75-c0f58519efefe08dc05934e99a59dce4.mat
  saved: /g/data/dx2/cl7720/AIFS/AIFS_marine/netcdf/20251228/20251228T00_tos.nc

======== 20251229 ========

[20251229] Processing tos
  saved: /g/data/dx2/cl7720/AIFS/AIFS_marine/netcdf/20251229/20251229T00_tos.nc

======== 20251230 ========

[20251230] Processing tos
  saved: /g